In [ ]:
import cv2
import os
import glob
import re
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Video, display

In [ ]:
# Input & Output Paths
input_folder = r"E:\Fatemeh\20250609-PH\S0"
output_video = r"E:\Fatemeh\20250609-PH\S0_Overlay\overlay_ratio_timelapse22.mp4"

# Video Settings
frame_rate = 30
skip_frames = 2
line_y_position = 1000  # Adjust this to the desired scanline

# Load and sort image files
image_files = glob.glob(os.path.join(input_folder, "*.tif")) + glob.glob(os.path.join(input_folder, "*.tiff"))

if os.path.exists(input_folder):
    print("Folder found!")
else:
    print("Error: Folder not found! Check the path.")

In [ ]:
def extract_time_index(filename):
    match = re.search(r"t(\d+)", filename)
    return int(match.group(1)) if match else float('inf')

image_files.sort(key=extract_time_index)

if not image_files:
    print("❌ No TIFF images found.")
    raise SystemExit()

# Temporary folder for plot images
temp_dir = os.path.join(input_folder, "_temp_profile_plots")
os.makedirs(temp_dir, exist_ok=True)

# Placeholder for video writer
video_writer = None
frame_count = 0


# 🔍 PREVIEW horizontal line position on first image
preview_img_path = image_files[0]
img_preview = cv2.imread(preview_img_path, cv2.IMREAD_GRAYSCALE)

if img_preview is None:
    print("❌ Could not load first image for preview.")
else:
    # Show image with horizontal line
    img_with_line = cv2.cvtColor(img_preview, cv2.COLOR_GRAY2RGB)
    cv2.line(img_with_line, (0, line_y_position), (img_with_line.shape[1]-1, line_y_position), (255, 0, 0), 2)

    # Display using matplotlib
    plt.figure(figsize=(10, 6))
    plt.imshow(img_with_line)
    plt.title(f"Preview of Horizontal Line at y = {line_y_position}")
    plt.axis("on")
    plt.show()

In [ ]:
# This code will overlay histogram on the imagee!!

# 🔁 Process images and add to video
for i, img_file in enumerate(image_files):
    if i % skip_frames != 0:
        continue

    if i % 20 == 0:
        print(f"🧪 Processing frame {i}/{len(image_files)}")

    # Load RAW grayscale image (used for profile)
    img_gray_raw = cv2.imread(img_file, cv2.IMREAD_GRAYSCALE)
    if img_gray_raw is None:
        print(f"⚠️ Skipping unreadable file: {img_file}")
        continue

    if line_y_position >= img_gray_raw.shape[0]:
        print(f"⚠️ Line y={line_y_position} is outside image bounds.")
        continue

    # Enhance image for display (without affecting profile)
    def adjust_gamma(image, gamma=1.2):
        inv_gamma = 1.0 / gamma
        table = np.array([(i / 255.0) ** inv_gamma * 255 for i in np.arange(256)]).astype("uint8")
        return cv2.LUT(image, table)

    img_gray_display = adjust_gamma(img_gray_raw, gamma=1.5)
    img_overlay = cv2.cvtColor(img_gray_display, cv2.COLOR_GRAY2BGR)

    # Extract intensity profile from raw image
    intensity_profile = img_gray_raw[line_y_position, :].astype(np.int32)

    # Draw intensity profile directly on the scanline (red, thicker)
    for x in range(1, len(intensity_profile)):
        y1 = line_y_position - int(intensity_profile[x - 1]*2)  # Scaled to fit in view: Each intensity value is scaled (divided by 4) to map 0–255 range to a ~60-pixel band.
        y2 = line_y_position - int(intensity_profile[x]*2)
        pt1 = (x - 1, y1)
        pt2 = (x, y2)
        cv2.line(img_overlay, pt1, pt2, (0, 0, 255), 4)

     # Optional: draw base scanline in gray for reference
    cv2.line(img_overlay, (0, line_y_position), (img_gray_raw.shape[1] - 1, line_y_position), (180, 180, 180), 1)

    # Initialize video writer if not done
    if video_writer is None:
        h, w = img_overlay.shape[:2]
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        video_writer = cv2.VideoWriter(output_video, fourcc, frame_rate, (w, h))

    video_writer.write(img_overlay)
    frame_count += 1

# 🔚 Finalize
video_writer.release()
cv2.destroyAllWindows()
print(f"✅ Video saved as: {output_video}")
print(f"🎞️ Total frames added: {frame_count}")

# Play video inside notebook
display(Video(output_video, embed=True))